# C02Shift Colab Quickstart

This is the first-time setup notebook. It prepares a Drive-backed workspace, downloads the required public Sleipner data from CO2DataShare, and runs the direct p10 gate first.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount('/content/drive')

GITHUB_REPO = 'https://github.com/TallowCatch/Propagation.git'
WORKSPACE = Path('/content/drive/MyDrive/Propagation')
print('Python:', sys.version)

In [ ]:
if not WORKSPACE.exists():
    print('Cloning repo into Drive workspace...')
    !git clone {GITHUB_REPO} {WORKSPACE}
else:
    print('Workspace already exists:', WORKSPACE)

os.chdir(WORKSPACE)
print('Current working directory:', Path.cwd())
if not (Path.cwd() / '.git').exists():
    raise RuntimeError('Workspace exists but is not a git checkout. Please fix WORKSPACE path.')

In [ ]:
import subprocess

def run(cmd: str):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, check=False, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())
    return result.returncode

status_code = run('git status --porcelain')
if status_code != 0:
    raise RuntimeError('git status failed')

dirty = subprocess.run('git status --porcelain', shell=True, check=False, text=True, capture_output=True).stdout.strip()
if dirty:
    print('Workspace has local changes. Stashing before pull...')
    run("git stash push -u -m 'colab-autostash-before-pull'")

code = run('git pull --ff-only')
if code != 0:
    print('Fast-forward pull failed. Falling back to fetch + reset to origin/main for clean Colab runs...')
    run('git fetch origin')
    code = run('git reset --hard origin/main')
    if code != 0:
        raise RuntimeError('Could not sync repository; please restart runtime and try again.')


In [ ]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -e .[viz]

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python colab/bootstrap_public_data.py --workspace .

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli run-all --config configs/paper_wave_temporal_colab.yaml

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli evaluate-field --config configs/sleipner_p10_wave_temporal_colab.yaml

In [ ]:
import json
from pathlib import Path

gate_path = Path('runs/sleipner_p10_wave_temporal_colab/results/summary.json')
if not gate_path.exists():
    raise FileNotFoundError(f'Missing p10 gate summary at {gate_path}')

gate = json.loads(gate_path.read_text())
overall = gate['Field']['volume_summary']['overall']
c02shift = overall.get('wave_temporal_constrained', {})

# Fixed reference from your strongest prior baseline run.
baseline_support_volume_iou = 0.6852598556479389
baseline_trace_iou_floor = 0.4321
baseline_outside_support_ceiling = 0.2959

print('C02Shift support-volume IoU (p10):', c02shift.get('support_volume_iou_2010'))
print('C02Shift trace-support IoU (p10):', c02shift.get('trace_iou_with_2010_support'))
print('C02Shift outside-support fraction (p10):', c02shift.get('predicted_fraction_outside_support_volume'))
print('C02Shift crossline continuity (p10):', c02shift.get('crossline_continuity'))

pass_support = c02shift.get('support_volume_iou_2010', -1) > baseline_support_volume_iou
pass_trace = c02shift.get('trace_iou_with_2010_support', -1) >= baseline_trace_iou_floor
pass_outside = c02shift.get('predicted_fraction_outside_support_volume', 1) <= baseline_outside_support_ceiling

print('Gate pass - support volume:', pass_support)
print('Gate pass - trace floor:', pass_trace)
print('Gate pass - outside-support ceiling:', pass_outside)
print('Overall direct p10 gate passed:', bool(pass_support and pass_trace and pass_outside))

## Optional next step
Run this only if the direct p10 gate is competitive.

In [ ]:
# !PYTHONPATH=src python -m ccs_monitoring.cli evaluate-field --config configs/sleipner_p07_wave_temporal_colab.yaml